# DVF 2022 — Format de la colonne Surface Carrez

Ce notebook examine le format de stockage de la colonne Surface Carrez du 1er lot. Il compare son type à celui de la surface réelle bâtie, puis affiche quelques valeurs brutes, afin d'observer comment la mesure est enregistrée dans le fichier.

## Cellule 1 — Connexion au fichier

Interrogation avec DuckDB, directement au format Parquet. Le chemin pointe vers le fichier `dvf-2022.parquet` placé dans le dossier `data/`.

In [1]:
import duckdb
from pathlib import Path

# Chemin vers le fichier Parquet, place dans le dossier data/
FICHIER = Path(r"./data/dvf-2022.parquet")

con = duckdb.connect()
pq = str(FICHIER)
assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
print(f"Fichier : {FICHIER.name}")

Fichier : dvf-2022.parquet


## Cellule 2 — Type des deux colonnes de surface

On compare le type de stockage de la Surface Carrez à celui de la surface réelle bâtie. Les deux mesurent une surface, mais ne sont pas forcément enregistrées dans le même format.

In [2]:
types = con.execute(f"""
    DESCRIBE SELECT "Surface reelle bati", "Surface Carrez du 1er lot"
    FROM '{pq}'
""").fetchdf()
types[["column_name", "column_type"]]

,column_name,column_type
0,Surface reelle bati,BIGINT
1,Surface Carrez du 1er lot,VARCHAR


## Cellule 3 — Valeurs brutes de la Surface Carrez

Affichage de dix valeurs de la colonne qui contiennent une virgule, telles qu'elles sont stockées, et comptage du nombre total de valeurs comportant une virgule décimale.

In [3]:
exemples = con.execute(f"""
    SELECT "Surface Carrez du 1er lot" AS carrez_brut
    FROM '{pq}'
    WHERE "Surface Carrez du 1er lot" IS NOT NULL
      AND "Surface Carrez du 1er lot" LIKE '%,%'
    LIMIT 10
""").fetchall()

print("Exemples de Surface Carrez (valeurs brutes) :")
for (v,) in exemples:
    print(f"  {v}")

nb_virgule = con.execute(f"""
    SELECT COUNT(*)
    FROM '{pq}'
    WHERE "Surface Carrez du 1er lot" LIKE '%,%'
""").fetchone()[0]
print(f"\nNombre de valeurs avec virgule decimale : {nb_virgule:,}".replace(',', ' '))

Exemples de Surface Carrez (valeurs brutes) :
  24,10
  123,23
  39,05
  39,05
  70,02
  70,02
  70,02
  9,80
  5,06
  64,92

Nombre de valeurs avec virgule decimale : 419 277
